In [1]:
!pip install groq -q

import os, json
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)
os.makedirs("agent", exist_ok=True)
print("Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00
Ready!


In [10]:
from google.colab import files
import shutil, os

os.makedirs("model", exist_ok=True)
os.makedirs("agent", exist_ok=True)

uploaded = files.upload()  # select ONE file at a time

for filename in uploaded:
    if filename.endswith(".pth"):
        shutil.copy(filename, "model/best_model.pth")
        size = os.path.getsize("model/best_model.pth") / 1024 / 1024
        print(f"✓ Model saved! Size: {size:.1f} MB")
    elif filename.endswith(".json"):
        shutil.copy(filename, f"agent/{filename}")
        print(f"✓ JSON saved: agent/{filename}")

Saving best_model.pth to best_model (1).pth
✓ Model saved! Size: 15.8 MB


In [11]:
os.makedirs("/root/.kaggle", exist_ok=True)
kaggle_credentials = {
    "username": "ishwarirautray",
    "key": "KGAT_c42443df0152824059832271ff3ac0e1"
}
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
import zipfile
with zipfile.ZipFile("plantvillage-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("plantvillage")
print("Dataset ready!")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
plantvillage-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset ready!


In [13]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load class names
base_path   = "plantvillage/plantvillage dataset/color"
dataset     = datasets.ImageFolder(base_path)
CLASS_NAMES = dataset.classes

# Load model
ml_model = efficientnet_b0(weights=None)
ml_model.classifier[1] = nn.Linear(1280, 38)
ml_model.load_state_dict(torch.load("model/best_model.pth", map_location=device))
ml_model = ml_model.to(device)
ml_model.eval()

print(f"Model ready on {device} | {len(CLASS_NAMES)} classes")

Model ready on cuda | 38 classes


In [14]:
final_agent_code = '''"""
agent.py — Crop Disease Detection Agent
Complete agent for use in Streamlit app (Week 6)

Usage:
    from agent.agent import CropDiseaseAgent
    agent = CropDiseaseAgent(api_key, model_path, data_dir)
    result = agent.diagnose_image(image_path)
    reply  = agent.chat("follow-up question")
    agent.reset()
"""

import os
import json
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
from PIL import Image
from groq import Groq


SYSTEM_PROMPT = """You are Dr. Krishi, an expert agricultural plant pathologist with 20 years
of field experience helping farmers across India and Southeast Asia.

Your role:
- Diagnose plant diseases accurately based on ML model predictions
- Give practical, affordable treatment advice farmers can act on immediately
- Speak in a warm, caring tone — farmers may be stressed about losing their crops
- Always mention severity clearly so farmers understand urgency
- Prioritise organic treatments first, then chemical as backup
- Use conversation history to answer follow-up questions naturally
- If you do not know something, say so honestly — never hallucinate

Never guess or hallucinate treatment names.
If a disease is outside your database, say so clearly and recommend consulting a local expert."""


class CropDiseaseAgent:
    def __init__(self, api_key, model_path, data_dir="agent",
                 class_names_path=None, device=None):
        # Groq client
        self.client = Groq(api_key=api_key)

        # Device
        self.device = device or (
            torch.device("cuda") if torch.cuda.is_available()
            else torch.device("cpu")
        )

        # Load class names
        if class_names_path:
            with open(class_names_path, "r") as f:
                self.class_names = json.load(f)
        else:
            self.class_names = self._default_class_names()

        # Load ML model
        self.model = efficientnet_b0(weights=None)
        self.model.classifier[1] = nn.Linear(1280, len(self.class_names))
        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )
        self.model = self.model.to(self.device)
        self.model.eval()

        # Data paths
        self.disease_data_path   = os.path.join(data_dir, "disease_data.json")
        self.treatment_data_path = os.path.join(data_dir, "treatment_data.json")

        # Session state
        self.conversation_history = []
        self.diagnosis_memory     = []

        # Image transform
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    # ------------------------------------------------------------------ #
    #  ML INFERENCE                                                        #
    # ------------------------------------------------------------------ #
    def predict(self, image_path):
        """Run ML model on image. Returns prediction dict."""
        if image_path is None:
            return {"error": "No image provided. Please upload a leaf photo."}

        try:
            img    = Image.open(image_path).convert("RGB")
            tensor = self.transform(img).unsqueeze(0).to(self.device)
        except Exception as e:
            return {"error": f"Could not open image: {str(e)}"}

        with torch.no_grad():
            outputs    = self.model(tensor)
            probs      = torch.softmax(outputs, dim=1)
            conf, idx  = probs.max(1)

        raw_name   = self.class_names[idx.item()]
        plant      = raw_name.split("___")[0].replace("_", " ")
        disease    = raw_name.split("___")[1].replace("_", " ")
        confidence = round(conf.item() * 100, 2)

        return {
            "plant":      plant,
            "disease":    disease,
            "confidence": confidence,
            "raw_name":   raw_name,
            "error":      None
        }

    # ------------------------------------------------------------------ #
    #  TOOLS                                                               #
    # ------------------------------------------------------------------ #
    def disease_info(self, disease_name):
        with open(self.disease_data_path, "r") as f:
            data = json.load(f)
        if disease_name in data:
            return {"disease": disease_name, **data[disease_name], "found": True}
        for key in data:
            if (key.lower() in disease_name.lower() or
                    disease_name.lower() in key.lower()):
                return {"disease": key, **data[key], "found": True}
        return {"disease": disease_name, "cause": "Unknown",
                "symptoms": "Unknown", "severity": "Unknown", "found": False}

    def treatment_advice(self, disease_name, farming_type="both"):
        with open(self.treatment_data_path, "r") as f:
            data = json.load(f)
        matched = None
        if disease_name in data:
            matched = disease_name
        else:
            for key in data:
                if (key.lower() in disease_name.lower() or
                        disease_name.lower() in key.lower()):
                    matched = key
                    break
        if not matched:
            return {"disease": disease_name,
                    "organic":     ["Consult local agricultural officer"],
                    "chemical":    ["Consult local agricultural officer"],
                    "prevention":  "No specific data available",
                    "found":       False}
        info   = data[matched]
        result = {"disease": matched, "prevention": info["prevention"], "found": True}
        if farming_type in ("organic", "both"):
            result["organic"]  = info["organic"]
        if farming_type in ("chemical", "both"):
            result["chemical"] = info["chemical"]
        return result

    # ------------------------------------------------------------------ #
    #  MEMORY                                                              #
    # ------------------------------------------------------------------ #
    def _add_to_memory(self, plant, disease, confidence):
        self.diagnosis_memory.append({
            "plant": plant, "disease": disease, "confidence": confidence
        })
        if len(self.diagnosis_memory) > 3:
            self.diagnosis_memory.pop(0)

    def _get_memory_context(self):
        if not self.diagnosis_memory:
            return "No previous diagnoses this session."
        ctx = "Previous diagnoses this session:\\n"
        for i, e in enumerate(self.diagnosis_memory, 1):
            ctx += f"{i}. {e[\'plant\']} — {e[\'disease\']} ({e[\'confidence\']}%)\\n"
        return ctx

    # ------------------------------------------------------------------ #
    #  GROQ CALLS                                                          #
    # ------------------------------------------------------------------ #
    def _get_messages(self, user_message):
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
        msgs.extend(self.conversation_history)
        msgs.append({"role": "user", "content": user_message})
        return msgs

    def _call_groq(self, user_message, max_tokens=400):
        messages = self._get_messages(user_message)
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=max_tokens
        )
        reply = response.choices[0].message.content
        self.conversation_history.append({"role": "user",      "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": reply})
        return reply

    # ------------------------------------------------------------------ #
    #  PUBLIC METHODS                                                      #
    # ------------------------------------------------------------------ #
    def diagnose_image(self, image_path, farming_type="both"):
        """
        Full pipeline: image → ML prediction → agent report.
        This is what Streamlit calls when user uploads a photo.
        """
        # Step 1 — ML prediction
        prediction = self.predict(image_path)
        if prediction.get("error"):
            return {"error": prediction["error"]}

        # Step 2 — Tool calls
        info      = self.disease_info(prediction["disease"])
        treatment = self.treatment_advice(prediction["disease"], farming_type)

        # Step 3 — Confidence note
        conf = prediction["confidence"]
        if conf < 80:
            conf_note = f"NOTE: Low confidence ({conf}%). Recommend visual confirmation."
        elif conf < 95:
            conf_note = f"Model confidence: {conf}% — good but not certain."
        else:
            conf_note = f"Model confidence: {conf}% — high confidence diagnosis."

        organic_str  = "\\n".join([f"  • {t}" for t in treatment.get("organic",  [])])
        chemical_str = "\\n".join([f"  • {t}" for t in treatment.get("chemical", [])])

        user_message = f"""
{self._get_memory_context()}
Plant: {prediction["plant"]} | Disease: {prediction["disease"]} | {conf_note}
Cause: {info.get("cause","Unknown")} | Symptoms: {info.get("symptoms","Unknown")}
Severity: {info.get("severity","Unknown")}
Organic treatments: {organic_str}
Chemical treatments: {chemical_str}
Prevention: {treatment.get("prevention","Monitor regularly")}
Farming preference: {farming_type}
Provide a complete diagnosis report as Dr. Krishi. Under 200 words.
"""
        report = self._call_groq(user_message, max_tokens=400)
        self._add_to_memory(
            prediction["plant"], prediction["disease"], prediction["confidence"]
        )

        return {
            "plant":      prediction["plant"],
            "disease":    prediction["disease"],
            "confidence": prediction["confidence"],
            "severity":   info.get("severity", "Unknown"),
            "report":     report,
            "organic":    treatment.get("organic",  []),
            "chemical":   treatment.get("chemical", []),
            "prevention": treatment.get("prevention", ""),
            "error":      None
        }

    def chat(self, user_question):
        """
        Follow-up Q&A. Farmer types a question after diagnosis.
        This is what Streamlit calls when user sends a chat message.
        """
        if not user_question or not user_question.strip():
            return "Please type a question and I will do my best to help."
        return self._call_groq(user_question, max_tokens=250)

    def reset(self):
        """Start a fresh session. Call when user uploads a new image."""
        self.conversation_history = []
        self.diagnosis_memory     = []

    def _default_class_names(self):
        return [
            "Apple___Apple_scab", "Apple___Black_rot",
            "Apple___Cedar_apple_rust", "Apple___healthy",
            "Blueberry___healthy", "Cherry_(including_sour)___Powdery_mildew",
            "Cherry_(including_sour)___healthy",
            "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
            "Corn_(maize)___Common_rust_", "Corn_(maize)___Northern_Leaf_Blight",
            "Corn_(maize)___healthy", "Grape___Black_rot",
            "Grape___Esca_(Black_Measles)",
            "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
            "Orange___Haunglongbing_(Citrus_greening)",
            "Peach___Bacterial_spot", "Peach___healthy",
            "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
            "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
            "Raspberry___healthy", "Soybean___healthy",
            "Squash___Powdery_mildew", "Strawberry___Leaf_scorch",
            "Strawberry___healthy", "Tomato___Bacterial_spot",
            "Tomato___Early_blight", "Tomato___Late_blight",
            "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot",
            "Tomato___Spider_mites Two-spotted_spider_mite",
            "Tomato___Target_Spot",
            "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
            "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
        ]
'''

with open("agent.py", "w") as f:
    f.write(final_agent_code)

print("agent.py written!")

agent.py written!


In [15]:
# Quick test using the class directly in notebook
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
from PIL import Image
from groq import Groq

# Manually replicate the class here for notebook testing
exec(open("agent.py").read())

agent = CropDiseaseAgent(
    api_key    = GROQ_API_KEY,
    model_path = "model/best_model.pth"
)

print(f"Agent created!")
print(f"Device: {agent.device}")
print(f"Classes: {len(agent.class_names)}")

Agent created!
Device: cuda
Classes: 38


In [16]:
import random
random.seed(7)

sample_indices = random.sample(range(len(dataset)), 5)

for i, idx in enumerate(sample_indices):
    img_path, true_idx = dataset.samples[idx]
    true_name = dataset.classes[true_idx].split("___")[1].replace("_", " ")

    print(f"\n{'='*65}")
    print(f"Image {i+1}/5 | True: {true_name}")
    print("="*65)

    result = agent.diagnose_image(img_path)

    if result.get("error"):
        print(f"ERROR: {result['error']}")
    else:
        match = "✓" if result["disease"] == true_name else "✗"
        print(f"Predicted: {result['disease']} ({result['confidence']}%) {match}")
        print(f"Severity:  {result['severity']}")
        print(f"\n{result['report']}")


Image 1/5 | True: Bacterial spot
Predicted: Bacterial spot (100.0%) ✓
Severity:  Medium

I'm so glad we could diagnose the issue with your peach plants. With 100% confidence, I can tell you that your plants are suffering from Bacterial spot, caused by the bacterium Xanthomonas arboricola pv. pruni. The symptoms you're seeing, such as small water-soaked spots on leaves turning brown and a shot-hole appearance, along with fruit lesions, are all consistent with this disease.

The severity is medium, so it's essential we act quickly to prevent further damage. I recommend starting with organic treatments: apply a copper-based bactericide at bud swell, remove and destroy any infected plant material, and avoid overhead irrigation to reduce leaf wetness. 

If the situation worsens, we can consider chemical treatments like Oxytetracycline or copper hydroxide sprays. For long-term prevention, planting resistant peach varieties and avoiding sites with frequent rain can help. Let's work together 

In [17]:
# Chat follow-up after last diagnosis
print("\nFollow-up Q&A after last diagnosis:")
print("="*65)

questions = [
    "What is the most important thing I should do today?",
    "Is this safe for organic farming?"
]

for q in questions:
    print(f"\nFarmer: {q}")
    print(f"Dr. Krishi: {agent.chat(q)}")


Follow-up Q&A after last diagnosis:

Farmer: What is the most important thing I should do today?
Dr. Krishi: Considering the diagnoses we've made so far, I would say the most important thing you should do today is to remove the infected lower leaves from your potato and tomato plants. This will help prevent the spread of Early blight and Septoria leaf spot, respectively, and give your plants a better chance of recovery.

Removing the infected leaves is a simple yet effective step that can be taken immediately, and it's a crucial part of the organic treatment approach we discussed. By doing this, you'll be able to prevent further damage and reduce the risk of the disease spreading to other parts of the plant.

Make sure to dispose of the removed leaves properly to prevent the spread of the disease, and then you can start implementing the other treatment and prevention strategies we discussed. How does that sound?

Farmer: Is this safe for organic farming?
Dr. Krishi: Removing infected 

In [18]:
agent.reset()
print(f"After reset:")
print(f"  Conversation history: {len(agent.conversation_history)} messages")
print(f"  Diagnosis memory:     {len(agent.diagnosis_memory)} entries")
print("Reset working correctly!")

After reset:
  Conversation history: 0 messages
  Diagnosis memory:     0 entries
Reset working correctly!


In [19]:
from google.colab import files
files.download("agent.py")
print("agent.py downloaded!")
print("Save it to your agent/ folder on laptop.")
print()
print("=== WEEK 5 COMPLETE ===")
print("Agent is fully built and tested.")
print("Next: Week 6 — Streamlit UI!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

agent.py downloaded!
Save it to your agent/ folder on laptop.

=== WEEK 5 COMPLETE ===
Agent is fully built and tested.
Next: Week 6 — Streamlit UI!
